# ANALYSE-08 : Ramsey et van der Waerden — l'énumération qui prouve, et le mur où elle s'arrête

Deux théorèmes exacts de
petite combinatoire — $R(3,3) = 6$ et $W(2,3) = 9$ — y sont prouvés **par épuisement
brutal** : énumérer toutes les colorations, compter les survivantes, constater zéro.
Puis le même geste est tendu vers $R(4,4)$, où il meurt sur un mur de $2^{153}$
colorations — et l'argument probabiliste d'Erdős (1947) prend le relais : quand
compter devient impossible, **majorer** suffit pour l'existence.

Règle du carnet, héritée de la série : chaque affirmation numérique est **mesurée par
la cellule qui la précède**, ou citée avec sa source.
## Navigation

| Notebook précédent | Notebook suivant |
|---|---|
| [ANALYSE-07 - Percolation critique](ANALYSE-07-Percolation-Critique.ipynb) | [ANALYSE-09 - Tuilage apériodique](ANALYSE-09-Tuilage-Aperiodique.ipynb) |

**Jumeau Rust** : [ANALYSE-08-Ramsey-VdW-Rust](ANALYSE-08-Ramsey-VdW-Rust.ipynb)

## Plan du carnet

1. **Section 1** — $R(3,3)$ par épuisement sur $K_4$, $K_5$, $K_6$, et la preuve classique (sans machine) de la borne supérieure.
2. **Section 2** — le pont comptage → existence : l'argument d'Erdős (1947), puis le mur de $R(4,4)$ chiffré.
3. **Section 3** — $W(2,3)$ par le même moteur d'énumération, appliqué à un autre objet.
4. **Section 4** — synthèse mesurée en un seul tableau, puis trois exercices.

**Convention d'encodage.** Une coloration des arêtes de $K_n$ est un tableau de $\binom{n}{2}$ bits — le bit de l'arête $(i,j)$ vaut 1 si elle est rouge, 0 si elle est bleue —, exactement la variable `bits` des cellules de code ; sur $[1,n]$, c'est un tableau de $n$ bits. **Prérequis** : dénombrement (coefficients binomiaux) et probabilités élémentaires (espérance, linéarité).

## 1. Ramsey $R(3,3)$ : l'énoncé, et la preuve par épuisement

**Énoncé.** Colorons chaque arête de $K_n$ (le graphe complet sur $n$ sommets) en
rouge ou bleu. $R(3,3)$ est le plus petit $n$ tel que **toute** coloration contienne un
triangle monochrome. La valeur $R(3,3) = 6$ se démontre en une soirée d'ordinateur :
énumérer les $2^{\binom{n}{2}}$ colorations et vérifier, pour chacune, ses
$\binom{n}{3}$ triangles.

La stratégie du carnet : monter $n = 4, 5, 6$ et regarder le décompte s'effondrer —
18 survivantes, puis 12, puis 0.

La borne supérieure $R(3,3) \le 6$ se démontre aussi à la main, par le principe des tiroirs : dans toute 2-coloration des arêtes de $K_6$, fixons un sommet $v$ ; ses 5 arêtes incidentes ne reçoivent que deux couleurs, donc au moins 3 d'entre elles — disons vers $a$, $b$, $c$ — ont la même couleur, mettons rouge. Si l'une des arêtes $ab$, $ac$, $bc$ est rouge, elle forme avec $v$ un triangle rouge ; sinon ces trois arêtes sont toutes bleues et forment un triangle bleu. Dans tous les cas, $K_6$ contient un triangle monochrome : $R(3,3) \le 6$. L'énumération ci-dessous retrouve ce verdict par la machine — et exhibe en outre les 12 témoins de $K_5$ qui donnent $R(3,3) > 5$.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Les comparaisons exactes alimentent le bilan de parité de la synthèse.
var exactChecks = new List<(string Section, string Name, string Calculated, string Expected, int Mismatch)>();
string Fmt(FormattableString text) => text.ToString(CultureInfo.InvariantCulture);
string InvariantValue<T>(T value) => value is IFormattable formatted
    ? formatted.ToString(null, CultureInfo.InvariantCulture)
    : Convert.ToString(value, CultureInfo.InvariantCulture) ?? string.Empty;

void CheckExact<T>(string section, string name, T calculated, T expected)
{
    bool equal = EqualityComparer<T>.Default.Equals(calculated, expected);
    string actualText = InvariantValue(calculated);
    string expectedText = InvariantValue(expected);
    exactChecks.Add((section, name, actualText, expectedText, equal ? 0 : 1));
    if (!equal)
        throw new InvalidOperationException($"ÉCART — {section}, {name} : calculé {actualText}, attendu {expectedText}.");
}

// Arêtes (i < j) de K_n ; la position dans cette liste indexe la coloration.
List<(int A, int B)> EdgesOf(int n)
{
    var edges = new List<(int A, int B)>();
    for (int i = 0; i < n; i++)
        for (int j = i + 1; j < n; j++)
            edges.Add((i, j));
    return edges;
}

// Pour suivre l'ordre de product((0, 1), repeat=...), le premier bit est celui de poids fort.
IEnumerable<int[]> Colorings(int length)
{
    if (length < 0 || length > 30)
        throw new ArgumentOutOfRangeException(nameof(length));
    int total = 1 << length;
    for (int mask = 0; mask < total; mask++)
    {
        var bits = new int[length];
        for (int i = 0; i < length; i++)
            bits[i] = (mask >> (length - 1 - i)) & 1;
        yield return bits;
    }
}

string BitWord(int[] bits) => new string(Array.ConvertAll(bits, bit => bit == 1 ? '1' : '0'));

// Associer à chaque triangle ses trois indices d'arêtes, une seule fois par graphe.
List<(int A, int B, int C, int AB, int AC, int BC)> TriangleIndices(
    int n, List<(int A, int B)> edges)
{
    var index = new int[n, n];
    for (int i = 0; i < edges.Count; i++)
        index[edges[i].A, edges[i].B] = i;
    var triangles = new List<(int A, int B, int C, int AB, int AC, int BC)>();
    for (int a = 0; a < n; a++)
        for (int b = a + 1; b < n; b++)
            for (int c = b + 1; c < n; c++)
                triangles.Add((a, b, c, index[a, b], index[a, c], index[b, c]));
    return triangles;
}

// Premier triangle monochrome de la coloration bits (aucun élément s'il n'y en a pas).
// L'itérateur permet aussi de poursuivre le parcours pour les lister tous.
IEnumerable<(int Color, int A, int B, int C)> MonochromeTriangles(
    int[] bits, List<(int A, int B, int C, int AB, int AC, int BC)> triangles)
{
    foreach (var t in triangles)
    {
        int color = bits[t.AB];
        if (color == bits[t.AC] && color == bits[t.BC])
            yield return (color, t.A, t.B, t.C);
    }
}

// Énumération exhaustive : nombre de colorations sans triangle monochrome + témoin.
(int EdgeCount, int Total, int AvoiderCount, string Witness) CountTriangleAvoiders(int n)
{
    var edges = EdgesOf(n);
    var triangles = TriangleIndices(n, edges);
    int total = 0;
    int count = 0;
    string witness = string.Empty;
    foreach (var bits in Colorings(edges.Count))
    {
        total++;
        if (MonochromeTriangles(bits, triangles).Any())
            continue;
        if (count == 0)
            witness = BitWord(bits);
        count++;
    }
    return (edges.Count, total, count, witness);
}

// Illustration 1 — K₄ : décompte exhaustif et premier témoin.
const int n4 = 4;
var k4 = CountTriangleAvoiders(n4);
CheckExact("Ramsey", "K₄ : arêtes", k4.EdgeCount, 6);
CheckExact("Ramsey", "K₄ : colorations visitées", k4.Total, 64);
CheckExact("Ramsey", "K₄ : évitantes", k4.AvoiderCount, 18);
CheckExact("Ramsey", "K₄ : premier témoin", k4.Witness, "001100");
CheckExact("Ramsey", "R(3,3) > 4", k4.AvoiderCount > 0, true);
Console.WriteLine(Fmt($"K_{n4} : {k4.EdgeCount} aretes, {k4.Total} colorations"));
Console.WriteLine(Fmt($"sans triangle monochrome : {k4.AvoiderCount} (exemple : {k4.Witness})"));

K_4 : 6 aretes, 64 colorations


sans triangle monochrome : 18 (exemple : 001100)


### Lecture du résultat

Sur les $64$ colorations de $K_4$, dix-huit échappent au triangle monochrome — donc
$R(3,3) > 4$. Rien de surprenant : avec six arêtes, une coloration a beaucoup de
liberté. Ce premier palier fixe le geste : **le comptage exact est la preuve**, pas
une estimation. La question est de savoir jusqu'où il monte.

In [2]:
// Illustration 2 — K₅ et la structure de ses évitantes
const int n5 = 5;
var k5 = CountTriangleAvoiders(n5);
CheckExact("Ramsey", "K₅ : arêtes", k5.EdgeCount, 10);
CheckExact("Ramsey", "K₅ : colorations visitées", k5.Total, 1024);
CheckExact("Ramsey", "K₅ : évitantes", k5.AvoiderCount, 12);
CheckExact("Ramsey", "R(3,3) > 5", k5.AvoiderCount > 0, true);
Console.WriteLine(Fmt($"K_{n5} : {k5.EdgeCount} aretes, {k5.Total} colorations, sans triangle monochrome : {k5.AvoiderCount}"));

// Structure des survivantes : le rouge est-il toujours un cycle C5 ?
var edges5 = EdgesOf(n5);
var triangles5 = TriangleIndices(n5, edges5);
int c5Count = 0;
foreach (var bits in Colorings(edges5.Count))
{
    if (MonochromeTriangles(bits, triangles5).Any())
        continue;
    var degrees = new int[n5];
    int redEdges = 0;
    for (int i = 0; i < edges5.Count; i++)
    {
        if (bits[i] != 1)
            continue;
        var (a, b) = edges5[i];
        redEdges++;
        degrees[a]++;
        degrees[b]++;
    }
    // Un graphe simple 2-régulier à cinq sommets ne peut pas se scinder en deux cycles.
    if (redEdges == n5 && degrees.All(degree => degree == 2))
        c5Count++;
}
int labeledC5 = Enumerable.Range(1, n5).Aggregate(1, (product, factor) => product * factor) / (2 * n5);
CheckExact("Ramsey", "K₅ : cycles C₅ rouges parmi les évitantes", c5Count, 12);
CheckExact("Ramsey", "K₅ : toutes les évitantes sont des cycles C₅ rouges", c5Count, k5.AvoiderCount);
CheckExact("Ramsey", "K₅ : décompte des cycles C₅ étiquetés", c5Count, labeledC5);
Console.WriteLine(Fmt($"survivantes dont le rouge est exactement un cycle C5 : {c5Count} / {k5.AvoiderCount}"));

K_5 : 10 aretes, 1024 colorations, sans triangle monochrome : 12


survivantes dont le rouge est exactement un cycle C5 : 12 / 12


### Lecture du résultat

Douze survivantes sur $1024$ : $R(3,3) > 5$. Et la structure est totale — la mesure le
dit : **chacune des 12 survivantes a pour rouge un cycle $C_5$ exactement** (cinq
arêtes, chaque sommet de degré 2), le bleu étant le $C_5$ complémentaire. Ce n'est pas
un hasard d'échantillon : 12 est précisément le nombre de cycles hamiltoniens étiquetés
de $K_5$, soit $\frac{5!}{2 \cdot 5} = 12$. Survivre à Ramsey sur $K_5$, c'est
**être** un double pentagone — la borne constructive de ce côté est un objet unique,
douze fois déguisé.

In [3]:
// Illustration 3 — K₆ et les triangles d’une coloration fixée
const int n6 = 6;
var k6 = CountTriangleAvoiders(n6);
CheckExact("Ramsey", "K₆ : arêtes", k6.EdgeCount, 15);
CheckExact("Ramsey", "K₆ : colorations visitées", k6.Total, 32768);
CheckExact("Ramsey", "K₆ : évitantes", k6.AvoiderCount, 0);
// K₅ fournit la borne inférieure par restriction, K₆ la borne supérieure par épuisement.
int ramsey33 = k5.AvoiderCount > 0 && k6.AvoiderCount == 0 ? n6 : 0;
CheckExact("Ramsey", "Seuil R(3,3)", ramsey33, 6);
int ramseyGrowth = k6.Total / k5.Total;
CheckExact("Ramsey", "Facteur de croissance K₆ / K₅", ramseyGrowth, 32);
CheckExact("Ramsey", "Reste du quotient K₆ / K₅", k6.Total % k5.Total, 0);
Console.WriteLine(Fmt($"K_{n6} : {k6.EdgeCount} aretes, {k6.Total} colorations, sans triangle monochrome : {k6.AvoiderCount}"));

// Ce que devient une coloration quelconque : ses triangles monochromes.
// Donnée d'entrée citée du jumeau Python, reprise telle quelle.
int[] bits = { 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0 }; // 15 bits arbitraires
var edges6 = EdgesOf(n6);
var triangles6 = TriangleIndices(n6, edges6);
CheckExact("Ramsey", "Longueur de la coloration citée", bits.Length, edges6.Count);
string bitsText = string.Join(", ", bits.Select(bit => bit.ToString(CultureInfo.InvariantCulture)));
CheckExact("Ramsey", "Coloration d'entrée citée", bitsText, "0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0");
var tris = MonochromeTriangles(bits, triangles6).ToArray();
var expectedTriangles6 = new[] { "bleu : (0, 1, 5)", "rouge : (0, 2, 3)", "bleu : (0, 4, 5)", "rouge : (1, 2, 4)" };
CheckExact("Ramsey", "Triangles monochromes de la coloration citée", tris.Length, expectedTriangles6.Length);
Console.WriteLine();
Console.WriteLine(Fmt($"coloration arbitraire ({bitsText}) : {tris.Length} triangles monochromes"));
int triangleIndex6 = 0;
foreach (var (color, a, b, c) in tris.Take(4))
{
    string colorName = color == 1 ? "rouge" : "bleu";
    string line = Fmt($"{colorName} : ({a}, {b}, {c})");
    CheckExact("Ramsey", Fmt($"Triangle monochrome n°{triangleIndex6 + 1}"), line, expectedTriangles6[triangleIndex6]);
    Console.WriteLine("  " + line);
    triangleIndex6++;
}

K_6 : 15 aretes, 32768 colorations, sans triangle monochrome : 0


coloration arbitraire (0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0) : 4 triangles monochromes


  bleu : (0, 1, 5)


  rouge : (0, 2, 3)


  bleu : (0, 4, 5)


  rouge : (1, 2, 4)


### Verdict — $R(3,3) = 6$

Zéro survivante sur $32768$ : **toute** coloration de $K_6$ porte un triangle
monochrome, et $K_5$ en évitait encore douze. L'énumération intégrale est une preuve
complète : $R(3,3) = 6$, exactement. Notons l'asymétrie du coût — $K_5$ a demandé
$1024$ vérifications, $K_6$ en demande $32768$ : un facteur 32 pour **un** sommet de
plus. Chaque palier multiplie le monde par $2^{n}$ environ. C'est la première fois que
le mur se montre, et il n'est pas encore haut.

## 2. Le pont comptage → existence : l'argument d'Erdős

Pour $R(4,4)$, la même preuve demanderait d'épuiser les colorations de $K_{18}$ (la
valeur exacte $R(4,4) = 18$ est connue — mais pas par épuisement naïf : la cellule
ci-dessous chiffre l'obstacle). L'argument probabiliste d'Erdős (1947) renverse la
question : pour montrer qu'une coloration **sans** $K_k$ monochrome existe, nul besoin
de la trouver — il suffit de montrer que les colorations qui en contiennent un sont
**moins nombreuses que le total**.

La forme générale : si $\binom{n}{k} \cdot 2^{1 - \binom{k}{2}} < 1$, alors
$R(k,k) > n$. La quantité de gauche majore le nombre **attendu** de copies
monochromes de $K_k$ dans une coloration aléatoire de $K_n$ ; si elle est inférieure
à 1, une coloration sans aucune copie existe forcément.

D'où vient la condition $\binom{n}{k} 2^{1-\binom{k}{2}} < 1$ ? Prenons une coloration aléatoire de $K_n$, chaque arête tirée rouge ou bleue avec probabilité $\frac{1}{2}$, indépendamment des autres. $K_n$ contient $\binom{n}{k}$ copies de $K_k$, et chacune est monochrome avec probabilité $2 \cdot 2^{-\binom{k}{2}} = 2^{1-\binom{k}{2}}$ (deux couleurs possibles, $\binom{k}{2}$ arêtes qui doivent toutes coïncider).

Par linéarité de l'espérance, le nombre espéré $X$ de copies monochromes de $K_k$ vaut $\binom{n}{k} 2^{1-\binom{k}{2}}$. Si cette espérance est strictement inférieure à 1, il existe forcément une coloration avec $X = 0$ — une variable aléatoire entière positive d'espérance $< 1$ prend la valeur 0 avec probabilité strictement positive — donc une coloration sans aucun $K_k$ monochrome, c'est-à-dire $R(k,k) > n$. C'est la **méthode du premier moment** : aucune hypothèse d'indépendance entre les événements « copie monochrome » n'est requise, seule compte la somme de leurs probabilités.

In [4]:
// Illustration 4 — La table d’Erdős pour k de 3 à 8
// Convention de math.comb : il n'y a aucun sous-ensemble de taille k si k > n.
BigInteger Binomial(int n, int k)
{
    if (k < 0 || k > n)
        return BigInteger.Zero;
    k = Math.Min(k, n - k);
    BigInteger value = BigInteger.One;
    for (int i = 1; i <= k; i++)
        value = value * (n - k + i) / i;
    return value;
}

// Comme dans l'affichage Python, l'exposant a au moins deux chiffres.
string Scientific(double value, int decimals) => value.ToString(
    "0." + new string('0', decimals) + "e+00", CultureInfo.InvariantCulture);

// Encadrements connus : survey dynamique DS1 de Radziszowski (Electronic J. Combinatorics).
// Données citées du jumeau Python, non recalculées ici.
var knownRamsey = new Dictionary<int, string>
{
    [3] = "6 (exact)", [4] = "18 (exact)", [5] = "[43, 48]",
    [6] = "[102, 165]", [7] = "[205, 540]", [8] = "[281, 1870]"
};

// Références de parité : seuils et affichages de la sortie Python, avec les tailles exactes.
var expectedErdos = new (int K, int LastPassing, int Space, string Display)[]
{
    (3,  3,         8, "8.00e+00"),
    (4,  6,        64, "6.40e+01"),
    (5, 11,      1024, "1.02e+03"),
    (6, 17,     32768, "3.28e+04"),
    (7, 27,   2097152, "2.10e+06"),
    (8, 42, 268435456, "2.68e+08")
};
var erdosRows = new List<(int K, int LastPassing, BigInteger Space)>();
Console.WriteLine("La colonne « R(k,k) connu » cite les encadrements DS1 repris du jumeau Python.");
Console.WriteLine(Fmt($"{"k",2} {"Erdos donne",13} {"R(k,k) connu",13} {"mur 2^C(k,2)",12}"));
for (int k = 3; k <= 8; k++)
{
    int edgeCount = (int)Binomial(k, 2);
    BigInteger space = BigInteger.One << edgeCount;
    BigInteger limit = space >> 1;
    // E[X] < 1 équivaut exactement à C(n,k) < 2^(C(k,2)-1).
    int n = 3;
    while (Binomial(n, k) < limit)
        n++;
    int lastPassing = n - 1;
    var expected = expectedErdos[k - 3];
    string label = Fmt($"k = {k}");
    string spaceText = Scientific((double)space, 2);
    CheckExact("Erdős", label + " : indice", k, expected.K);
    CheckExact("Erdős", label + " : dernier n admissible", lastPassing, expected.LastPassing);
    CheckExact("Erdős", label + " : inégalité au dernier n admissible", Binomial(lastPassing, k) < limit, true);
    CheckExact("Erdős", label + " : inégalité au n suivant", Binomial(n, k) < limit, false);
    CheckExact("Erdős", label + " : taille exacte de l’espace", space, new BigInteger(expected.Space));
    CheckExact("Erdős", label + " : taille à la précision Python", spaceText, expected.Display);
    // En passant de k-1 à k, le graphe complet gagne k-1 arêtes.
    if (erdosRows.Count > 0)
        CheckExact("Erdős", label + " : croissance du mur", space, erdosRows[erdosRows.Count - 1].Space << (k - 1));
    erdosRows.Add((k, lastPassing, space));
    string boundText = Fmt($"R > {lastPassing}");
    Console.WriteLine(Fmt($"{k,2} {boundText,13} {knownRamsey[k],13} {spaceText,12}"));
}
CheckExact("Erdős", "Borne à k = 3 plus faible que par épuisement", erdosRows[0].LastPassing < n5 && k5.AvoiderCount > 0, true);

La colonne « R(k,k) connu » cite les encadrements DS1 repris du jumeau Python.


 k   Erdos donne  R(k,k) connu mur 2^C(k,2)


 3         R > 3     6 (exact)     8.00e+00


 4         R > 6    18 (exact)     6.40e+01


 5        R > 11      [43, 48]     1.02e+03


 6        R > 17    [102, 165]     3.28e+04


 7        R > 27    [205, 540]     2.10e+06


 8        R > 42   [281, 1870]     2.68e+08


### Lecture du résultat

La borne donne $R(3,3) > 3$ — médiocre : l'épuisement de la section 1 a donné $> 5$.
Mais regardez la colonne du mur : le monde à énumérer double à chaque palier de $k$
croissant, et l'encadrement connu s'élargit en proportion ($[43, 48]$ pour $k = 5$ :
cinq valeurs possibles, personne ne sait trancher). La borne probabiliste est
**faible en petite dimension** — elle ne remplace pas l'énumération là où
l'énumération est possible — et c'est exactement pourquoi les deux coexistent : le
comptage exact règne jusqu'où il atteint, l'argument de comptage **majoré** règne
au-delà.

In [5]:
// Illustration 5 — Le mur de R(4,4) : que coûterait l'énumération naïve de K_18 ?
// La taille de l'espace est calculée en entier exact avant la conversion en durée.
const int wallN = 18; // Valeur connue citée du jumeau Python, et non prouvée ici.
int wallEdges = (int)Binomial(wallN, 2);
BigInteger wallColorings = BigInteger.One << wallEdges;
const double rate = 1e9; // 1 milliard de colorations par seconde (optimiste).
double seconds = (double)wallColorings / rate;
double years = seconds / (365.25 * 24 * 3600);
const double ageUniverse = 1.38e10; // Années : donnée citée du jumeau Python.
double universeRatio = years / ageUniverse;

// Comparaison exacte des chaînes à la précision de la sortie Python (écart d'affichage nul exigé).
string rateText = rate.ToString("0e+0", CultureInfo.InvariantCulture).Replace("e+", "e");
string wallText = Scientific((double)wallColorings, 3);
string yearsText = Scientific(years, 2);
string ageText = Scientific(ageUniverse, 2);
string ratioText = Scientific(universeRatio, 1);
CheckExact("Mur", "K₁₈ : arêtes", wallEdges, 153);
CheckExact("Mur", "K₁₈ : taille exacte de l’espace", wallColorings,
    BigInteger.Parse("11417981541647679048466287755595961091061972992", CultureInfo.InvariantCulture));
CheckExact("Mur", "Cadence supposée : affichage", rateText, "1e9");
CheckExact("Mur", "K₁₈ : taille à la précision Python", wallText, "1.142e+46");
CheckExact("Mur", "Durée en années à la précision Python", yearsText, "3.62e+29");
CheckExact("Mur", "Âge de l’univers cité : affichage", ageText, "1.38e+10");
CheckExact("Mur", "Rapport à la précision Python", ratioText, "2.6e+19");
Console.WriteLine("Données reprises du jumeau Python : taille du graphe, cadence supposée et âge de l'univers.");
Console.WriteLine(Fmt($"colorations de K_{wallN} : 2^{wallEdges} = {wallText}"));
Console.WriteLine(Fmt($"a {rateText} colorations/s : {yearsText} annees"));
Console.WriteLine(Fmt($"age de l'univers    : {ageText} annees"));
Console.WriteLine(Fmt($"rapport             : {ratioText} x l'age de l'univers"));

Données reprises du jumeau Python : taille du graphe, cadence supposée et âge de l'univers.


colorations de K_18 : 2^153 = 1.142e+46


a 1e9 colorations/s : 3.62e+29 annees


age de l'univers    : 1.38e+10 annees


rapport             : 2.6e+19 x l'age de l'univers


### Verdict — le pont comptage → existence

Même à un milliard de colorations par seconde, épuiser $K_{18}$ demande de l'ordre de
$10^{19}$ âges de l'univers (rapport exact imprimé ci-dessus) : le geste qui a prouvé
$R(3,3) = 6$ est **mort** à partir de là. L'inégalité d'Erdős — une ligne — donne
quant à elle $R(k,k) > n$ pour tout $n$ vérifiant la condition : c'est le pont. Ce que
l'énumération obtenait par constat exhaustif (« zéro survivante »), la probabilistique
l'obtient par majoration. La valeur exacte $R(4,4) = 18$ a été établie en 1955 par
Greenwood et Gleason — par une preuve combinatoire, pas par force brute.

## 3. Van der Waerden $W(2,3)$ : le même geste, un autre objet

**Énoncé.** Colorons chaque entier de $[1, n]$ en rouge ou bleu. $W(2,3)$ est le plus
petit $n$ tel que **toute** coloration contienne une progression arithmétique
monochrome de longueur 3 — trois entiers $a, a+d, a+2d$ de la même couleur. Le
comptage est le même qu'en section 1 ($2^n$ colorations à vérifier une à une) ; seul
le motif cherché change : la structure passe du graphe à la droite arithmétique.

Le théorème de van der Waerden (1927) généralise le constat : pour tout nombre de couleurs $r$ et toute longueur $k$, il existe un plus petit entier $W(r,k)$ tel que toute $r$-coloration de $[1, W(r,k)]$ contienne une progression arithmétique monochrome de longueur $k$. Ici $r = 2$, $k = 3$ : une 3-AP est un trio $a, a+d, a+2d$ — par exemple $1, 2, 3$ avec $d = 1$, ou $1, 5, 9$ avec $d = 4$.

Le moteur est strictement le même qu'en section 1 — énumérer les $2^m$ colorations de $[1..m]$ et tester un prédicat — seule la définition du prédicat change : `TryMonochromeAp` cherche une 3-AP monochrome là où `MonochromeTriangles` cherchait un triangle. La cellule ci-dessous liste d'abord les colorations de $[1..8]$ qui évitent toute 3-AP monochrome ; la cellule suivante en fait le décompte pour chaque $m$ de 1 à 9.

In [6]:
// Illustration 6 — Les évitantes de [1..8]
// Première progression arithmétique monochrome de longueur 3 (false sinon).
// Comme en Python, le motif renvoyé est indexé à partir de zéro.
bool TryMonochromeAp(int[] coloring, out (int A, int B, int C) progression)
{
    int m = coloring.Length;
    for (int a = 0; a < m; a++)
    {
        for (int d = 1; a + 2 * d < m; d++)
        {
            int b = a + d;
            int c = a + 2 * d;
            if (coloring[a] == coloring[b] && coloring[a] == coloring[c])
            {
                progression = (a, b, c);
                return true;
            }
        }
    }
    progression = default;
    return false;
}

// Le même parcours fournit la liste ordonnée et le nombre de colorations visitées.
(int Total, List<string> Avoiders) ScanApColorings(int m)
{
    int total = 0;
    var avoiders = new List<string>();
    foreach (var coloring in Colorings(m))
    {
        total++;
        if (!TryMonochromeAp(coloring, out _))
            avoiders.Add(BitWord(coloring));
    }
    return (total, avoiders);
}

const int m8 = 8;
var ap8 = ScanApColorings(m8);
var expectedAp8 = new[] { "00110011", "01011010", "01100110", "10011001", "10100101", "11001100" };
CheckExact("Van der Waerden", "[1..8] : colorations visitées", ap8.Total, 256);
CheckExact("Van der Waerden", "[1..8] : évitantes", ap8.Avoiders.Count, expectedAp8.Length);
for (int i = 0; i < ap8.Avoiders.Count; i++)
    CheckExact("Van der Waerden", Fmt($"[1..8] : évitante n°{i + 1}"), ap8.Avoiders[i], expectedAp8[i]);

// Une répétition de période 4 a deux moitiés identiques.
var ap8Irregular = ap8.Avoiders.Where(word => !word.Take(4).SequenceEqual(word.Skip(4))).ToArray();
int irregularAp8 = ap8Irregular.Length;
int periodicAp8 = ap8.Avoiders.Count - irregularAp8;
CheckExact("Van der Waerden", "[1..8] : répétitions de période 4", periodicAp8, 4);
CheckExact("Van der Waerden", "[1..8] : autres formes", irregularAp8, 2);
string firstIrregularComplement = new string(ap8Irregular[0].Select(bit => bit == '0' ? '1' : '0').ToArray());
CheckExact("Van der Waerden", "[1..8] : formes irrégulières complémentaires", firstIrregularComplement, ap8Irregular[1]);
Console.WriteLine(Fmt($"[1..{m8}] : {ap8.Total} colorations, sans 3-AP monochrome : {ap8.Avoiders.Count}"));
foreach (var word in ap8.Avoiders)
    Console.WriteLine("  " + word);

[1..8] : 256 colorations, sans 3-AP monochrome : 6


  00110011


  01011010


  01100110


  10011001


  10100101


  11001100


### Lecture du résultat

Six survivantes sur $256$. Et la mesure dit mieux que le compte : **quatre** d'entre
elles sont des répétitions d'un motif de période 4 — `0011`, `0110` et leurs
compléments `1001`, `1100`. Les **deux** dernières, `01011010` et `10100101`
(compléments l'une de l'autre), ne sont **pas** périodiques : la contrainte laisse
donc passer aussi des formes irrégulières. C'est l'analogue du $C_5$ de la
section 1 : la contrainte ne laisse passer qu'un **petit nombre de formes** —
quatre régulières, deux irrégulières.

In [7]:
// Illustration 7 — Les évitantes de m = 1 à 9
// La chute : nombre d'évitantes pour chaque taille m = 1..9.
var expectedApSweep = new (int M, int Total, int AvoiderCount)[]
{
    (1,   2,  2), (2,   4,  4), (3,   8,  6),
    (4,  16, 10), (5,  32, 14), (6,  64, 20),
    (7, 128, 16), (8, 256,  6), (9, 512,  0)
};
var apSweep = new List<(int M, int Total, int AvoiderCount)>();
for (int m = 1; m <= 9; m++)
{
    var scan = ScanApColorings(m);
    int avoiderCount = scan.Avoiders.Count;
    var expected = expectedApSweep[m - 1];
    string label = Fmt($"m = {m}");
    CheckExact("Van der Waerden", label + " : taille", m, expected.M);
    CheckExact("Van der Waerden", label + " : colorations visitées", scan.Total, expected.Total);
    CheckExact("Van der Waerden", label + " : évitantes", avoiderCount, expected.AvoiderCount);
    if (apSweep.Count > 0)
    {
        var previous = apSweep[apSweep.Count - 1];
        CheckExact("Van der Waerden", label + " : doublement des colorations", scan.Total, 2 * previous.Total);
        CheckExact("Van der Waerden", label + " : sens de variation des évitantes",
            Math.Sign(avoiderCount - previous.AvoiderCount), m <= 6 ? 1 : -1);
    }
    apSweep.Add((m, scan.Total, avoiderCount));
    Console.WriteLine(Fmt($"m = {m} : {scan.Total,4} colorations -> {avoiderCount,3} evitantes"));
}

// Le premier zéro donne le seuil ; le témoin de la taille précédente donne la borne inférieure.
var apAtThreshold = apSweep.First(row => row.AvoiderCount == 0);
int vdw23 = apAtThreshold.M;
var apBeforeThreshold = apSweep.Single(row => row.M == vdw23 - 1);
CheckExact("Van der Waerden", "Seuil W(2,3)", vdw23, 9);
CheckExact("Van der Waerden", "W(2,3) > 8", apBeforeThreshold.AvoiderCount > 0, true);
CheckExact("Van der Waerden", "Taille de la liste détaillée avant le seuil", apBeforeThreshold.M, m8);
CheckExact("Van der Waerden", "Total avant le seuil : liste et balayage", apBeforeThreshold.Total, ap8.Total);
CheckExact("Van der Waerden", "Évitantes avant le seuil : liste et balayage",
    apBeforeThreshold.AvoiderCount, ap8.Avoiders.Count);

m = 1 :    2 colorations ->   2 evitantes


m = 2 :    4 colorations ->   4 evitantes


m = 3 :    8 colorations ->   6 evitantes


m = 4 :   16 colorations ->  10 evitantes


m = 5 :   32 colorations ->  14 evitantes


m = 6 :   64 colorations ->  20 evitantes


m = 7 :  128 colorations ->  16 evitantes


m = 8 :  256 colorations ->   6 evitantes


m = 9 :  512 colorations ->   0 evitantes


### Lecture du résultat — la frontière exacte

La population croît d'abord (20 évitantes à $m = 6$), plafonne, puis s'effondre :
16, 6, et **zéro** à $m = 9$. Chaque entier ajouté apporte de nouvelles progressions
— donc de nouvelles contraintes — mais seulement un facteur 2 de liberté ; au-delà
d'un seuil, les contraintes gagnent. $W(2,3) = 9$ : la preuve est, comme pour
$R(3,3) = 6$, un épuisement — 512 colorations vérifiées une à une, aucune n'y
échappe. Deux mondes (graphes, arithmétique), la même méthode, deux certitudes
exactes.

## 4. Synthèse — deux certitudes exactes, un mur, un pont

Cette section ramasse en un seul tableau les cinq mesures du carnet : pour chaque monde — $K_4$, $K_5$, $K_6$, $[1..8]$, $[1..9]$ — le nombre total de colorations énumérées et le nombre de survivantes. Le coût se lit dans la colonne « taille », le rendement dans « survivantes » : c'est le contraste entre un coût qui double à chaque palier et un rendement qui s'effondre à zéro qui constitue la preuve.

In [8]:
// Illustration 8 — Récapitulatif mesuré et attestation de parité
// Récapitulatif mesuré dans ce carnet : réutiliser les décomptes, sans réénumérer.
string LowerBoundLabel(string quantity, int size, int avoiderCount) => avoiderCount > 0
    ? Fmt($"{quantity} > {size}")
    : "Borne inférieure non établie";

// Le complément d'un C₅ dans K₅ est aussi un C₅ : il est lui-même simple et 2-régulier.
string k5Structure = c5Count == k5.AvoiderCount ? " (toutes des doubles C5)" : string.Empty;
var summaryRows = new (string Result, string World, int Total, string Survivors)[]
{
    (LowerBoundLabel("R(3,3)", n4, k4.AvoiderCount), Fmt($"K_{n4}"), k4.Total, Fmt($"{k4.AvoiderCount}")),
    (LowerBoundLabel("R(3,3)", n5, k5.AvoiderCount), Fmt($"K_{n5}"), k5.Total, Fmt($"{k5.AvoiderCount}{k5Structure}")),
    (Fmt($"R(3,3) = {ramsey33}"), Fmt($"K_{n6}"), k6.Total, Fmt($"{k6.AvoiderCount}")),
    (LowerBoundLabel("W(2,3)", apBeforeThreshold.M, apBeforeThreshold.AvoiderCount),
        Fmt($"[1..{apBeforeThreshold.M}]"), apBeforeThreshold.Total,
        Fmt($"{apBeforeThreshold.AvoiderCount} ({periodicAp8} periodiques + {irregularAp8} irregulieres)")),
    (Fmt($"W(2,3) = {vdw23}"), Fmt($"[1..{apAtThreshold.M}]"), apAtThreshold.Total,
        Fmt($"{apAtThreshold.AvoiderCount}"))
};

// Sortie du jumeau Python citée comme référence de contrôle, non utilisée pour produire le tableau.
var expectedSummary = new (string Result, string World, int Total, string Survivors)[]
{
    ("R(3,3) > 4", "K_4",     64, "18"),
    ("R(3,3) > 5", "K_5",   1024, "12 (toutes des doubles C5)"),
    ("R(3,3) = 6", "K_6",  32768, "0"),
    ("W(2,3) > 8", "[1..8]", 256, "6 (4 periodiques + 2 irregulieres)"),
    ("W(2,3) = 9", "[1..9]", 512, "0")
};
CheckExact("Synthèse", "Nombre d'instances récapitulées", summaryRows.Length, expectedSummary.Length);
Console.WriteLine(Fmt($"{"resultat",-12} {"monde",-8} {"taille",7}   survivantes"));
for (int i = 0; i < summaryRows.Length; i++)
{
    var row = summaryRows[i];
    var expected = expectedSummary[i];
    string label = Fmt($"Ligne {i + 1}");
    CheckExact("Synthèse", label + " : verdict", row.Result, expected.Result);
    CheckExact("Synthèse", label + " : monde", row.World, expected.World);
    CheckExact("Synthèse", label + " : colorations visitées", row.Total, expected.Total);
    CheckExact("Synthèse", label + " : survivantes", row.Survivors, expected.Survivors);
    Console.WriteLine(Fmt($"{row.Result,-12} {row.World,-8} {row.Total,7}   {row.Survivors}"));
}

// Compter les écarts dans le registre des assertions réellement exécutées.
Console.WriteLine();
Console.WriteLine("Attestation de parité — égalité exacte, à la précision affichée pour les notations scientifiques.");
foreach (var section in exactChecks.GroupBy(check => check.Section))
{
    int mismatches = section.Sum(check => check.Mismatch);
    string sectionStatus = mismatches == 0 ? "CONFORME" : "ÉCART";
    Console.WriteLine(Fmt($"{section.Key} — {sectionStatus} : {mismatches} écart(s) sur {section.Count()} comparaisons"));
    foreach (var check in section)
    {
        string status = check.Mismatch == 0 ? "CONFORME" : "ÉCART";
        Console.WriteLine(Fmt($"  {status} — {check.Name} : calculé {check.Calculated} ; attendu {check.Expected}"));
    }
}

resultat     monde     taille   survivantes


R(3,3) > 4   K_4           64   18


R(3,3) > 5   K_5         1024   12 (toutes des doubles C5)


R(3,3) = 6   K_6        32768   0


W(2,3) > 8   [1..8]       256   6 (4 periodiques + 2 irregulieres)


W(2,3) = 9   [1..9]       512   0


... (156 outputs left out)

  CONFORME — Ligne 5 : verdict : calculé W(2,3) = 9 ; attendu W(2,3) = 9


  CONFORME — Ligne 5 : monde : calculé [1..9] ; attendu [1..9]


  CONFORME — Ligne 5 : colorations visitées : calculé 512 ; attendu 512


  CONFORME — Ligne 5 : survivantes : calculé 0 ; attendu 0


### Verdict de synthèse

L'énumération a livré deux théorèmes ($R(3,3) = 6$, $W(2,3) = 9$) et des bornes
constructives en témoin ($R(3,3) > 5$, $W(2,3) > 8$) — à coût dérisoire. Le même
geste sur $R(4,4)$ se heurte à $2^{153}$ : le mur. Le pont d'Erdős remplace « toutes
vérifiées » par « moins nombreuses que le tout », et l'existence repose sur une
inégalité au lieu d'un constat. C'est la leçon de structure : **le comptage exact et
la majoration probabiliste ne sont pas deux méthodes concurrentes mais deux régimes du
même geste** — l'un règne tant que le monde tient en mémoire, l'autre prend le relais
et ne rend plus la valeur exacte, seulement un encadrement ($43 \le R(5,5) \le 48$ à
ce jour, survey DS1 de Radziszowski).

## Exercices

Trois exercices, tous sur le même moteur que les sections 1 à 3. Les stubs s'exécutent
sans erreur ; à vous de remplir le calcul.

Chaque exercice réutilise les fonctions définies dans les sections 1 à 3, sans introduire de nouveau concept : le moteur d'énumération est le même, seul le motif cherché change — un cycle $C_4$ rouge parmi les survivantes de $K_4$, un trio monochrome $x+y=z$ (Schur), ou la borne d'Erdős prolongée au-delà de $k = 8$.

### Exercice 1 — $K_4$ : les 18 survivantes, et leurs cycles

La section 1 a compté 18 colorations de $K_4$ sans triangle monochrome. Parmi elles,
combien ont pour rouge exactement un cycle $C_4$ (quatre arêtes rouges, chaque sommet
de degré rouge 2) ? Vérifiez votre réponse contre la valeur théorique : le nombre de
cycles de longueur 4 étiquetés sur 4 sommets.

In [9]:
// Exercice 1 — Cycles rouges parmi les évitantes de K₄
// TODO étudiant
int? resultatCyclesRouges = null;
_ = resultatCyclesRouges;
Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 2 — Schur : le troisième monde

Colorons $[1, n]$ en rouge et bleu ; une solution **monochrome** de $x + y = z$ est un
trio d'une même couleur. Quelles colorations de $[1..4]$ et de $[1..5]$ évitent tout trio
monochrome ? Même moteur d'énumération, troisième structure
— l'addition.

In [10]:
// Exercice 2 — Colorations sans trio monochrome x + y = z
IReadOnlyList<int[]> SchurAvoiders(int m)
{
    // TODO étudiant
    return Array.Empty<int[]>();
}

Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 3 — Jusqu'où l'inégalité d'Erdős grimpe-t-elle ?

La section 2 s'arrête à $k = 8$. Prolongez la table : $k = 9$ à $12$, borne d'Erdős
d'un côté ; de l'autre, l'encadrement courant de $R(k,k)$ tel que publié dans la
survey dynamique DS1 de Radziszowski (*Small Ramsey Numbers*, Electronic Journal of
Combinatorics — allez le chercher : c'est l'exercice de lecture). Que fait le
**rapport** entre la borne d'Erdős et la meilleure borne inférieure connue quand $k$
croît ?

In [11]:
// Exercice 3 — Table d’Erdős pour k de 9 à 12
// TODO étudiant
var bornesExercice = Array.Empty<(int K, int Borne)>();
_ = bornesExercice;
Console.WriteLine("Exercice a completer");

Exercice a completer


## Conclusion

Ce carnet a prouvé deux théorèmes en comptant tout, mesuré le mur où ce geste meurt
($2^{153}$ colorations pour $K_{18}$), et traversé ce mur par l'inégalité d'Erdős.
Trois structures — triangle de graphe, progression arithmétique, équation additive —
un seul moteur : énumérer, compter, conclure.

Références :

- **Erdős, P. (1947).** *Some remarks on the theory of graphs.* Bulletin of the
  American Mathematical Society. — l'argument probabiliste fondateur.
- **Greenwood, R. E. & Gleason, A. M. (1955).** *Combinatorial relations and
  chromatic graphs.* Canadian Journal of Mathematics. — $R(3,3) = 6$, $R(4,4) = 18$
  établis combinatoirement.
- **Radziszowski, S.** *Small Ramsey Numbers.* Electronic Journal of Combinatorics,
  Dynamic Survey DS1 (révisions successives). — l'état courant des encadrements.
- **van der Waerden, B. L. (1927).** *Beweis einer Baudetschen Vermutung.* Nieuw
  Archief voor Wiskunde. — le théorème, dans sa formulation d'origine.

> Version C# du [carnet Python ANALYSE-08](ANALYSE-08-Ramsey-VdW.ipynb).